# Nifty 50 Signal — Training v2 (quantile labels + class weights)

**v1 outcome:** 43% accuracy with the meta-learner predicting NO_TRADE almost always.
**v2 changes:**
1. **Quantile labels** — bottom 33% of next-bar returns = BUY_PUT, top 33% = BUY_CALL, middle = NO_TRADE. Forces a balanced 33/33/33 class split on any timeframe.
2. **Class weights** — XGBoost (sample_weight), LightGBM (`class_weight='balanced'`), CatBoost (`auto_class_weights='Balanced'`).
3. **Logistic Regression meta** — Random Forest meta overfit on 1k rows. LR is more honest with limited data.

Setup: Runtime → T4 GPU. CSVs at `/MyDrive/nifty_signal/data/`.

In [ ]:
!pip install -q ta xgboost lightgbm catboost scikit-learn

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DATA_DIR = '/content/drive/MyDrive/nifty_signal/data'
MODELS_DIR = '/content/drive/MyDrive/nifty_signal/models'
os.makedirs(MODELS_DIR, exist_ok=True)
print('Files in data dir:', os.listdir(DATA_DIR))

In [ ]:
import pandas as pd, numpy as np
from ta.momentum import RSIIndicator
from ta.trend import MACD, EMAIndicator
from ta.volatility import AverageTrueRange, BollingerBands

FEATURES = [
    'rsi_14','macd_hist','ema_9','ema_21','ema_50',
    'ema_cross_9_21','ema_cross_21_50','atr_14','bb_pct_b',
    'pcr','oi_change_pct','dte','day_of_week','session_flag'
]

def build(df, label_mode='quantile', q_low=0.33, q_high=0.67):
    """Engineer features + label. label_mode: 'quantile' (balanced) or 'fixed' (0.5% threshold)."""
    df = df.copy()
    df.columns = [c.strip().lower() for c in df.columns]
    if 'datetime' not in df.columns:
        for c in ('date','timestamp','time'):
            if c in df.columns: df = df.rename(columns={c:'datetime'}); break
    c, h, l = df['close'], df['high'], df['low']
    df['rsi_14'] = RSIIndicator(c, 14).rsi()
    df['macd_hist'] = MACD(c, 26, 12, 9).macd_diff()
    df['ema_9'] = EMAIndicator(c, 9).ema_indicator()
    df['ema_21'] = EMAIndicator(c, 21).ema_indicator()
    df['ema_50'] = EMAIndicator(c, 50).ema_indicator()
    df['ema_cross_9_21'] = (df['ema_9'] > df['ema_21']).astype(int)
    df['ema_cross_21_50'] = (df['ema_21'] > df['ema_50']).astype(int)
    df['atr_14'] = AverageTrueRange(h, l, c, 14).average_true_range()
    bb = BollingerBands(c, 20, 2)
    df['bb_pct_b'] = (c - bb.bollinger_lband()) / (bb.bollinger_hband() - bb.bollinger_lband())
    if 'oi_call' in df.columns and 'oi_put' in df.columns:
        df['pcr'] = (df['oi_put'] / df['oi_call'].replace(0, np.nan)).fillna(1.0)
        df['oi_change_pct'] = (df['oi_call'] + df['oi_put']).pct_change().fillna(0.0)
    else:
        df['pcr'] = 1.0
        df['oi_change_pct'] = 0.0
    dt = pd.to_datetime(df['datetime'])
    df['day_of_week'] = dt.dt.dayofweek
    df['dte'] = (3 - dt.dt.dayofweek) % 7
    m = dt.dt.hour*60 + dt.dt.minute
    df['session_flag'] = ((m >= 570) & (m <= 660)).astype(int)
    nxt = c.shift(-1)
    if label_mode == 'quantile':
        ret = (nxt - c) / c
        lo, hi = ret.quantile(q_low), ret.quantile(q_high)
        lab = pd.Series(0, index=df.index)
        lab[ret > hi] = 1
        lab[ret < lo] = -1
        print(f'  quantile thresholds: low={lo*100:.3f}%  high={hi*100:.3f}%')
    else:
        lab = pd.Series(0, index=df.index)
        lab[nxt > c*1.005] = 1
        lab[nxt < c*0.995] = -1
    df['label'] = lab
    return df.dropna().reset_index(drop=True)

raw = pd.read_csv(f'{DATA_DIR}/nifty50_daily.csv')
feats = build(raw, label_mode='quantile')
X = feats[FEATURES]
y = feats['label'].map({-1:0, 0:1, 1:2}).astype(int)
print(f'usable rows: {X.shape[0]}')
print(f'class distribution: {y.value_counts().sort_index().to_dict()}  (0=BUY_PUT, 1=NO_TRADE, 2=BUY_CALL)')

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
import joblib

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, shuffle=False)
print(f'train: {X_tr.shape[0]}  |  test: {X_te.shape[0]}')
print(f'train class dist: {pd.Series(y_tr).value_counts().sort_index().to_dict()}')
print(f'test class dist:  {pd.Series(y_te).value_counts().sort_index().to_dict()}')

sw_tr = compute_sample_weight('balanced', y_tr)

print('\n== XGBoost ==')
xgb = XGBClassifier(n_estimators=400, max_depth=5, learning_rate=0.05,
                    objective='multi:softprob', num_class=3, eval_metric='mlogloss',
                    tree_method='hist', device='cuda')
xgb.fit(X_tr, y_tr, sample_weight=sw_tr)
print('acc:', accuracy_score(y_te, xgb.predict(X_te)))
print(classification_report(y_te, xgb.predict(X_te), target_names=['BUY_PUT','NO_TRADE','BUY_CALL'], digits=3))
joblib.dump(xgb, f'{MODELS_DIR}/xgboost.pkl')

print('== LightGBM ==')
lgbm = LGBMClassifier(n_estimators=400, learning_rate=0.05, objective='multiclass',
                     num_class=3, class_weight='balanced', verbose=-1)
lgbm.fit(X_tr, y_tr)
print('acc:', accuracy_score(y_te, lgbm.predict(X_te)))
print(classification_report(y_te, lgbm.predict(X_te), target_names=['BUY_PUT','NO_TRADE','BUY_CALL'], digits=3))
joblib.dump(lgbm, f'{MODELS_DIR}/lightgbm.pkl')

print('== CatBoost ==')
cat = CatBoostClassifier(iterations=400, depth=6, learning_rate=0.05,
                         loss_function='MultiClass', auto_class_weights='Balanced',
                         task_type='GPU', verbose=0)
cat.fit(X_tr, y_tr)
print('acc:', accuracy_score(y_te, cat.predict(X_te)))
print(classification_report(y_te, cat.predict(X_te), target_names=['BUY_PUT','NO_TRADE','BUY_CALL'], digits=3))
joblib.dump(cat, f'{MODELS_DIR}/catboost.pkl')

In [ ]:
print('== Stacking meta (Logistic Regression) ==')
base_tr = np.hstack([xgb.predict_proba(X_tr), lgbm.predict_proba(X_tr), cat.predict_proba(X_tr)])
base_te = np.hstack([xgb.predict_proba(X_te), lgbm.predict_proba(X_te), cat.predict_proba(X_te)])

meta = LogisticRegression(max_iter=1000, class_weight='balanced', C=0.5)
meta.fit(base_tr, y_tr)
preds = meta.predict(base_te)
print('Ensemble accuracy:', accuracy_score(y_te, preds))
print(classification_report(y_te, preds, target_names=['BUY_PUT','NO_TRADE','BUY_CALL'], digits=3))
joblib.dump(meta, f'{MODELS_DIR}/meta_rf.pkl')  # keep filename for backend compatibility

# Per-class probability summary so we can see if the model has any signal
import numpy as np
proba_te = meta.predict_proba(base_te)
print('\nMean predicted prob by true class:')
for cls, name in enumerate(['BUY_PUT','NO_TRADE','BUY_CALL']):
    mask = y_te.values == cls
    if mask.any():
        print(f'  true={name}: prob_buy_put={proba_te[mask,0].mean():.3f}  prob_no_trade={proba_te[mask,1].mean():.3f}  prob_buy_call={proba_te[mask,2].mean():.3f}')

## Optional — LSTM on hourly with quantile labels
Now that labels are balanced 33/33/33 on hourly too, the LSTM should actually learn something.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input

hourly = pd.read_csv(f'{DATA_DIR}/nifty50_hourly.csv')
hf = build(hourly, label_mode='quantile')
Xh = hf[FEATURES].values
yh = hf['label'].map({-1:0, 0:1, 1:2}).astype(int).values
print(f'hourly class dist: {pd.Series(yh).value_counts().sort_index().to_dict()}')

WINDOW = 60
Xs, ys = [], []
for i in range(WINDOW, len(Xh)):
    Xs.append(Xh[i-WINDOW:i])
    ys.append(yh[i])
Xs, ys = np.array(Xs), np.array(ys)
split = int(len(Xs)*0.8)
print(f'sequences: {Xs.shape}  | train: {split}  | test: {len(Xs)-split}')

from sklearn.utils.class_weight import compute_class_weight
cw = compute_class_weight('balanced', classes=np.arange(3), y=ys[:split])
cw_dict = {i: w for i, w in enumerate(cw)}
print(f'class weights: {cw_dict}')

model = Sequential([
    Input(shape=(WINDOW, Xh.shape[1])),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(32),
    Dense(3, activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.fit(Xs[:split], ys[:split], epochs=10, batch_size=128,
          validation_data=(Xs[split:], ys[split:]), class_weight=cw_dict)
model.save(f'{MODELS_DIR}/lstm_model.h5')

# Verify it's actually predicting all three classes
lstm_preds = model.predict(Xs[split:], verbose=0).argmax(axis=1)
print(f'\nLSTM prediction distribution on test: {pd.Series(lstm_preds).value_counts().sort_index().to_dict()}')
print(classification_report(ys[split:], lstm_preds, target_names=['BUY_PUT','NO_TRADE','BUY_CALL'], digits=3))

In [ ]:
for f in sorted(os.listdir(MODELS_DIR)):
    size_mb = os.path.getsize(f'{MODELS_DIR}/{f}') / 1e6
    print(f'  {f}  ({size_mb:.2f} MB)')